# Electricity Load Forecasting — Baseline + SHAP-Guided Feature Engineering

This notebook covers, end to end and inline (no external scripts):

1. Load raw data, chronological train/test split (train: 2016–2025, test: 2026)
2. Baseline feature engineering (calendar, weather, lagged load only)
3. Baseline XGBoost training + evaluation (global + Peak-MAPE)
4. SHAP diagnostics — global and peak-only
5. Diagnose under-prediction on peak hours
6. Iteration zone — add SHAP-informed features here, one round at a time
7. Ablation testing zone

**Discipline for iterating in this notebook:** change one feature (or one small
group) at a time in Section 6, re-run from that cell down, and re-check both
SHAP rankings and Peak-MAPE before adding the next one. Don't add everything
at once — you won't know which change did what.


## 0. Imports & config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from pandas.tseries.holiday import USFederalHolidayCalendar

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (14, 5)

RANDOM_STATE = 42
TRAIN_END = "2025-12-31 23:00:00"   # inclusive
TEST_START = "2026-01-01 00:00:00"


## 1. Load raw data & chronological split

`df_raw` is assumed already available with columns:
`['timestamp', 'actual_load_mw', 'temp_c', 'humidity_pct', 'precip_mm']`

If it isn't in memory yet, load it below (adjust path).


In [ ]:
# df_raw = pd.read_parquet("path/to/raw_data.parquet")

df_raw = df_raw.copy()
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)

print(df_raw.shape)
print(df_raw["timestamp"].min(), "→", df_raw["timestamp"].max())
df_raw.head()


In [ ]:
df_train_raw = df_raw[df_raw["timestamp"] <= TRAIN_END].reset_index(drop=True)
df_test_raw  = df_raw[df_raw["timestamp"] >= TEST_START].reset_index(drop=True)

print(f"Train: {len(df_train_raw):,} rows | {df_train_raw['timestamp'].min()} → {df_train_raw['timestamp'].max()}")
print(f"Test:  {len(df_test_raw):,} rows | {df_test_raw['timestamp'].min()} → {df_test_raw['timestamp'].max()}")


## 2. Baseline feature engineering

Baseline = calendar features + basic weather + two lagged-load features only.
Nothing SHAP-informed yet — this is the plain reference model.

We only have hourly `temp_c` (not the paper's daily station tmax/tmin), so
`tmax`/`tmin`/`tavg` are derived as rolling 24h extrema/mean of `temp_c`,
**shifted by 1 hour** so no leakage of the current hour's temperature.

Applied identically to train and test — same code, run on each frame
separately, nothing is fit on train and applied to test yet since none of
these baseline features need fitting (no learned thresholds at this stage).


In [ ]:
def add_baseline_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- calendar features (from timestamp only) ---
    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"].dt.month
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

    # --- weather features (derived from temp_c) ---
    # shift(1) first so the 24h window never includes the current hour
    temp_shifted = df["temp_c"].shift(1)
    df["tavg"] = temp_shifted.rolling(24, min_periods=24).mean()
    df["tmax"] = temp_shifted.rolling(24, min_periods=24).max()
    df["tmin"] = temp_shifted.rolling(24, min_periods=24).min()
    df["prcp"] = df["precip_mm"]  # kept as-is, paper's naming

    # --- lagged load features ---
    df["load_lag_24"]  = df["actual_load_mw"].shift(24)
    df["load_lag_168"] = df["actual_load_mw"].shift(168)

    return df


df_train_base = add_baseline_features(df_train_raw)
df_test_base  = add_baseline_features(df_test_raw)

df_train_base.tail()


### Drop warm-up NaNs (rows where lags/rolling windows aren't full yet)

In [ ]:
BASELINE_FEATURES = [
    "hour", "dayofweek", "month", "is_weekend",
    "tavg", "tmax", "tmin", "prcp",
    "load_lag_24", "load_lag_168",
]

df_baseline = df_train_base.dropna(subset=BASELINE_FEATURES + ["actual_load_mw"]).reset_index(drop=True)
print(f"Rows after dropping NaN warm-up: {len(df_baseline):,}")
print(f"Date range: {df_baseline['timestamp'].iloc[0]} → {df_baseline['timestamp'].iloc[-1]}")

df_test_baseline = df_test_base.dropna(subset=BASELINE_FEATURES + ["actual_load_mw"]).reset_index(drop=True)
print(f"Test rows after dropping NaN warm-up: {len(df_test_baseline):,}")


## 3. Train baseline XGBoost

In [ ]:
X_train = df_baseline[BASELINE_FEATURES]
y_train = df_baseline["actual_load_mw"]

X_test = df_test_baseline[BASELINE_FEATURES]
y_test = df_test_baseline["actual_load_mw"]

baseline_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)


## 4. Evaluate baseline — global + Peak-MAPE

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return 100 * np.mean(np.abs((y_true - y_pred) / y_true))

def peak_mape(y_true, y_pred, pctile=0.95):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    threshold = np.quantile(y_true, pctile)
    mask = y_true >= threshold
    return mape(y_true[mask], y_pred[mask]), mask.sum(), threshold

def evaluate(y_true, y_pred, label=""):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    m = mape(y_true, y_pred)
    pm, n_peak, thresh = peak_mape(y_true, y_pred)
    print(f"── {label} ──")
    print(f"  MAE       : {mae:10.2f} MW")
    print(f"  RMSE      : {rmse:10.2f} MW")
    print(f"  MAPE      : {m:8.4f} %")
    print(f"  Peak-MAPE : {pm:8.4f} %  (top 5%, n={n_peak}, threshold≥{thresh:.0f} MW)")
    return {"mae": mae, "rmse": rmse, "mape": m, "peak_mape": pm}

baseline_metrics = evaluate(y_test.values, y_pred_baseline, "BASELINE (test)")


In [ ]:
# Quick visual: actual vs predicted over the test period
fig, ax = plt.subplots()
ax.plot(df_test_baseline["timestamp"], y_test, label="Actual", alpha=0.7)
ax.plot(df_test_baseline["timestamp"], y_pred_baseline, label="Predicted (baseline)", alpha=0.7)
ax.set_title("Baseline XGBoost — Actual vs Predicted Load (2026 test)")
ax.legend()
plt.show()


## 5. SHAP diagnostics — baseline model

Two passes:
- **Global**: which features matter most on average, across all of test.
- **Peak-only**: which features matter most specifically on the top-5%
  highest-load hours — this is the ranking that tells you what to fix.


In [ ]:
explainer = shap.TreeExplainer(baseline_model)
shap_values_test = explainer(X_test)

shap.summary_plot(shap_values_test, X_test, show=True)


In [ ]:
peak_threshold = np.quantile(y_test, 0.95)
peak_mask = y_test.values >= peak_threshold
print(f"Peak subset: {peak_mask.sum()} rows, threshold ≥ {peak_threshold:.0f} MW")

X_test_peak = X_test[peak_mask]
shap_values_peak = explainer(X_test_peak)

shap.summary_plot(shap_values_peak, X_test_peak, show=True)


In [ ]:
# Ranked importance tables (global vs peak-only) side by side for easy comparison
def shap_ranking(shap_values, feature_names):
    mean_abs = np.abs(shap_values.values).mean(axis=0)
    return (pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
              .sort_values("mean_abs_shap", ascending=False)
              .reset_index(drop=True))

global_rank = shap_ranking(shap_values_test, BASELINE_FEATURES)
peak_rank = shap_ranking(shap_values_peak, BASELINE_FEATURES)

comparison = global_rank.merge(peak_rank, on="feature", suffixes=("_global", "_peak"))
comparison


## 6. Diagnose under-prediction on peak hours

Look at the *error*, not just the ranking — is the baseline systematically
underpredicting the highest-load hours? This is the pattern SHAP-informed
features are meant to fix.


In [ ]:
residuals_peak = y_test.values[peak_mask] - y_pred_baseline[peak_mask]

fig, ax = plt.subplots()
ax.scatter(df_test_baseline["timestamp"][peak_mask], residuals_peak, alpha=0.6, s=15)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Baseline residuals (actual − predicted) on peak hours")
ax.set_ylabel("Residual (MW) — positive = underprediction")
plt.show()

print(f"Mean residual on peak hours: {residuals_peak.mean():.1f} MW "
      f"({'underpredicting' if residuals_peak.mean() > 0 else 'overpredicting'} on average)")


---
## 7. Iteration zone — SHAP-informed feature engineering (v2)

Based on Sections 5–6 findings, add ONE round of new features at a time here,
re-run from this cell down, and re-check the SHAP rankings + Peak-MAPE before
adding the next round. Don't add everything from the paper at once — you
want to know which specific feature moved the needle on *your* data.

Candidate features to try (add incrementally, not all at once):
- `load_spike_vs_mean = (load_lag_24 - load_roll_mean_24) / (load_roll_mean_24 + 1)`
- `temp_spike_vs_mean = (tmax - tavg) / (tavg + 1)`
- `CDD`, `HDD` (degree days from `tavg`) and `CDD_x_hour`
- `lag_24_x_hour = load_lag_24 * hour`
- `is_extreme_heat_event` (tmax above a learned/fixed threshold)
- `load_roll_mean_24`, `load_roll_std_24`, `load_roll_mean_168`

Remember: any feature using `load_roll_mean_24` etc. should be built from
*lagged* load values, never the current-hour actual, or you'll leak the
answer into the input.


In [ ]:
def add_round_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add one round of SHAP-informed features here. Start with just one
    or two candidates, re-run the pipeline, check SHAP + Peak-MAPE, then
    come back and add the next round below."""
    df = df.copy()

    # --- round 1: example, uncomment / adjust as you decide what to try ---
    # df["load_roll_mean_24"] = df["actual_load_mw"].shift(1).rolling(24, min_periods=24).mean()
    # df["load_spike_vs_mean"] = (
    #     (df["load_lag_24"] - df["load_roll_mean_24"]) / (df["load_roll_mean_24"] + 1)
    # )

    return df


df_train_v2 = add_round_features(df_train_base)
df_test_v2  = add_round_features(df_test_base)

# update this list as you add features above
ROUND_FEATURES = BASELINE_FEATURES + [
    # "load_spike_vs_mean",
]

df_v2 = df_train_v2.dropna(subset=ROUND_FEATURES + ["actual_load_mw"]).reset_index(drop=True)
df_test_v2_clean = df_test_v2.dropna(subset=ROUND_FEATURES + ["actual_load_mw"]).reset_index(drop=True)

print(f"Train rows: {len(df_v2):,} | Test rows: {len(df_test_v2_clean):,} | Features: {len(ROUND_FEATURES)}")


In [ ]:
X_train_v2 = df_v2[ROUND_FEATURES]
y_train_v2 = df_v2["actual_load_mw"]
X_test_v2 = df_test_v2_clean[ROUND_FEATURES]
y_test_v2 = df_test_v2_clean["actual_load_mw"]

model_v2 = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1,
)
model_v2.fit(X_train_v2, y_train_v2)
y_pred_v2 = model_v2.predict(X_test_v2)

v2_metrics = evaluate(y_test_v2.values, y_pred_v2, "ROUND v2 (test)")

print("\n── IMPROVEMENT vs BASELINE ──")
for k in ["mape", "peak_mape", "mae", "rmse"]:
    delta = v2_metrics[k] - baseline_metrics[k]
    print(f"  Δ{k:10s}: {delta:+.4f}  ({'better' if delta < 0 else 'worse'})")


In [ ]:
explainer_v2 = shap.TreeExplainer(model_v2)
shap_values_v2 = explainer_v2(X_test_v2)
shap.summary_plot(shap_values_v2, X_test_v2, show=True)

peak_mask_v2 = y_test_v2.values >= np.quantile(y_test_v2, 0.95)
shap_values_peak_v2 = explainer_v2(X_test_v2[peak_mask_v2])
shap.summary_plot(shap_values_peak_v2, X_test_v2[peak_mask_v2], show=True)


---
## 8. Ablation testing zone

For features that ranked high in Section 7's SHAP output, remove one at a
time and re-measure Peak-MAPE — this separates "genuinely useful" from
"correlated with something already in the model."


In [ ]:
def ablate(features_to_drop: list, X_train_full, y_train_full, X_test_full, y_test_full, label=""):
    keep = [f for f in X_train_full.columns if f not in features_to_drop]
    m = xgb.XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X_train_full[keep], y_train_full)
    preds = m.predict(X_test_full[keep])
    return evaluate(y_test_full.values, preds, label=f"ABLATION: dropped {features_to_drop}")

# example usage once you have engineered features to test:
# ablate(["load_spike_vs_mean"], X_train_v2, y_train_v2, X_test_v2, y_test_v2)
